# Fine-tuning preliminar — extração de LME (Qwen + LoRA/Unsloth)

**Objetivo:** ajustar um modelo pequeno (Qwen 0.5B ou 1.5B) para transformar um
texto livre de solicitação de LME em JSON com `medicamento` e `quantidade`.

**Importante:** este notebook é um roteiro. Ele **não foi executado** durante a
montagem da entrega. Todos os números de treino e avaliação devem ser copiados
da execução real no Colab para `resultados.md`; não preencha valores por
estimativa.

**Pré-requisitos**

- Conta Google e Google Colab com GPU T4 (Ambiente de execução > Alterar tipo de
  ambiente de execução > T4 GPU).
- Os arquivos `train.jsonl` e `test.jsonl` gerados por `gerar_dataset.py`.
- O arquivo `avaliar.py` desta mesma pasta.

**Ordem:** execute as células de cima para baixo. Cada célula tem um comentário
explicando o que ela faz.

## 1. Instalar as dependências

A instalação pode levar alguns minutos. Se o comando do Unsloth falhar, consulte
o comando atualizado em <https://docs.unsloth.ai/> e mantenha as mesmas versões
de `trl`, `peft`, `accelerate`, `bitsandbytes` e `transformers` que o Unsloth
indicar.

In [ ]:
%%capture
# Instala Unsloth e as bibliotecas de treino. O %%capture esconde o log longo.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

## 2. Conferir a GPU

A saída deve mostrar uma GPU Tesla T4. Se aparecer “no GPU”, volte em
*Ambiente de execução > Alterar tipo de ambiente de execução* e escolha **T4
GPU**.

In [ ]:
# Mostra a GPU disponível e a memória livre.
!nvidia-smi

## 3. Enviar o dataset

Execute a célula e selecione `train.jsonl` e `test.jsonl` (os dois de uma vez).
Se preferir, monte o Google Drive e copie os arquivos para `/content/`.

In [ ]:
from google.colab import files

# Selecione train.jsonl e test.jsonl na janela de upload.
enviados = files.upload()
print("Arquivos enviados:", list(enviados))

In [ ]:
# Confere as contagens reais do dataset que você enviou.
import json

def carregar(caminho):
    with open(caminho, encoding="utf-8") as arquivo:
        return [json.loads(linha) for linha in arquivo if linha.strip()]

treino = carregar("train.jsonl")
teste = carregar("test.jsonl")
print("Treino:", len(treino))
print("Teste:", len(teste))
print("Exemplo:", json.dumps(treino[0], ensure_ascii=False, indent=2))

## 4. Carregar o modelo base

Use `Qwen/Qwen2.5-0.5B-Instruct` para o experimento mais rápido. Se a T4
aguentar e você quiser comparar, troque para `Qwen/Qwen2.5-1.5B-Instruct`.
Mantenha o mesmo nome no comando de avaliação no final.

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024
model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # ou Qwen/Qwen2.5-1.5B-Instruct

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,          # deixa o Unsloth escolher (fp16/bf16)
    load_in_4bit=True,   # cabe na T4
)

## 5. Adicionar o adaptador LoRA

O LoRA treina apenas uma parte pequena dos pesos. Isso reduz memória e tempo e
gera um adaptador que pode ser salvo separadamente do modelo base.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 6. Formatar o dataset

Cada exemplo vira uma conversa `user` (instrução + texto) e `assistant` (JSON
esperado). O `apply_chat_template` usa o formato de chat do Qwen, o mesmo que
será usado na avaliação.

In [ ]:
from datasets import Dataset

def formatar(exemplo):
    mensagens = [
        {
            "role": "user",
            "content": exemplo["instruction"] + "\n\n" + exemplo["input"],
        },
        {"role": "assistant", "content": exemplo["output"]},
    ]
    texto = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": texto}

dataset = Dataset.from_list([formatar(exemplo) for exemplo in treino])
print("Exemplos formatados:", len(dataset))
print(dataset[0]["text"][:500])

## 7. Configurar o treino

Hiperparâmetros iniciais para um experimento preliminar na T4: 3 épocas,
batch 2 por dispositivo, acumulação de 4, learning rate 2e-4. Se ocorrer erro
de memória, reduza `per_device_train_batch_size` para 1 e aumente
`gradient_accumulation_steps` para 8.

> Se a versão do TRL instalada reclamar do argumento `tokenizer`, troque por
> `processing_class=tokenizer`. Se `dataset_text_field` não for aceito, mova-o
> para `SFTConfig(..., dataset_text_field="text")`.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

## 8. Treinar

A célula abaixo executa o treino e imprime o tempo e a loss registrados pelo
Trainer. Copie esses valores reais para `resultados.md`.

In [ ]:
trainer_stats = trainer.train()
print("Tempo de treino (s):", trainer_stats.metrics.get("train_runtime"))
print("Loss final:", trainer_stats.metrics.get("train_loss"))

## 9. Salvar o adaptador

O adaptador fica em `lme-qwen-lora/`. O modelo base não é duplicado. Se quiser
guardar no Google Drive, execute a célula opcional.

In [ ]:
model.save_pretrained("lme-qwen-lora")
tokenizer.save_pretrained("lme-qwen-lora")
print("Adaptador salvo em lme-qwen-lora/")

In [ ]:
# Opcional: copiar o adaptador para o Google Drive.
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r lme-qwen-lora /content/drive/MyDrive/lme-qwen-lora

## 10. Testar uma inferência

Esta célula usa o primeiro exemplo de teste. A resposta deve ser um JSON com
`medicamento` e `quantidade`.

In [ ]:
FastLanguageModel.for_inference(model)

exemplo = teste[0]
mensagens = [
    {
        "role": "user",
        "content": exemplo["instruction"] + "\n\n" + exemplo["input"],
    }
]
entrada = tokenizer.apply_chat_template(
    mensagens, tokenize=False, add_generation_prompt=True
)
ids = tokenizer(entrada, return_tensors="pt").to("cuda")
saida = model.generate(**ids, max_new_tokens=64, do_sample=False)
resposta = tokenizer.decode(
    saida[0][ids["input_ids"].shape[1]:], skip_special_tokens=True
)
print("Entrada:", exemplo["input"])
print("Esperado:", exemplo["output"])
print("Obtido:", resposta)

## 11. Avaliar base vs. ajustado nos 30 exemplos

Envie o `avaliar.py` desta pasta quando a célula pedir. O script avalia o modelo
base e, em seguida, o mesmo modelo com o adaptador LoRA. Ele imprime a tabela com
JSON válido, medicamento, quantidade e acerto completo.

Se o adaptador não carregar, confira se `model_name` no treino é exatamente o
mesmo `--base-model` da avaliação.

In [ ]:
from google.colab import files

print("Envie o arquivo avaliar.py (pasta finetuning/)")
files.upload()

In [ ]:
!python avaliar.py   --base-model Qwen/Qwen2.5-0.5B-Instruct   --adapter lme-qwen-lora   --test test.jsonl   --limite 30   --saida resultados_avaliacao.json

## 12. Registrar os resultados

Copie a tabela impressa e os tempos de treino para `finetuning/resultados.md`,
substituindo os campos `PENDENTE`. Não invente nem arredonde para “melhorar” o
resultado: use exatamente a saída do Colab.

Se quiser comparar 0.5B e 1.5B, repita o notebook mudando `model_name` e
`--base-model`, e registre as duas linhas separadamente.

## Limitações

- O dataset é sintético e pequeno; a acurácia no teste não prova desempenho em
  textos reais.
- O teste tem apenas 30 exemplos, então cada acerto vale 3,3 pontos
  percentuais; diferenças pequenas entre modelos não são conclusivas.
- O modelo pode decorar formatos do dataset. A avaliação mede extração no mesmo
  domínio sintético, não generalização clínica.
- O experimento não substitui avaliação com dados reais, revisão humana nem
  avaliação de privacidade/LGPD.